# 🧪 Product Matching System — Backend Experiments

Use this notebook to test and validate every part of the matching pipeline **before running the Streamlit app**.

## Notebook Structure
| Section | What it tests |
|---|---|
| 0 | Setup, path fix & HF token loading |
| 1 | Data loading & inspection |
| 2 | Text cleaning |
| 3 | Model loading & embeddings |
| 4 | FAISS index |
| 5 | Single product matching |
| 6 | Direct URL-to-URL comparison |
| 7 | Domain filtering |
| 8 | Batch processing |
| 9 | Accuracy evaluation (ground truth) |
| 10 | AHT benchmarking |

---
## Section 0 — Setup, Path Fix & HF Token

> ⚠️ **Run this cell FIRST every time before anything else.**  
> It fixes the Python path AND loads your `.env` token — HuggingFace checks for the token at import time, so this must run before any model imports.

In [1]:
import sys
import os

# ── Reliable project root detection ───────────────────────────────────
# Walks upward from current directory until it finds the utils/ folder.
# Works whether you launch Jupyter from project root OR a subfolder.

project_root = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(project_root, 'utils')):
        break
    project_root = os.path.dirname(project_root)

if not os.path.isdir(os.path.join(project_root, 'utils')):
    raise RuntimeError(
        'Could not find utils/ folder.\n'
        'Launch Jupyter from inside the project folder:\n'
        '   cd product_matching_system_Basemodel\n'
        '   jupyter notebook'
    )

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root : {project_root}')
print(f'utils/ found : {os.path.isdir(os.path.join(project_root, "utils"))}')
print(f'data/  found : {os.path.isdir(os.path.join(project_root, "data"))}')

Project root : c:\DSAI\ProductMatchAnalysis\product_matching_system_Basemodel
utils/ found : True
data/  found : True


In [2]:
# ── Load .env and set HF_TOKEN BEFORE importing HuggingFace ───────────
#
# Why order matters:
#   HuggingFace checks for the token at import time.
#   If you import sentence_transformers first, the warning is already
#   printed and cannot be suppressed. So .env MUST be loaded first.

try:
    from dotenv import load_dotenv
    env_path = os.path.join(project_root, '.env')
    if os.path.exists(env_path):
        load_dotenv(env_path)
        print(f'✅ .env loaded from : {env_path}')
    else:
        print(f'❌ .env NOT found at: {env_path}')
        print('   Create a .env file in the project root containing:')
        print('   HF_TOKEN=hf_your_token_here')
except ImportError:
    print('❌ python-dotenv not installed')
    print('   Run:  pip install python-dotenv')
    print('   Then restart kernel and run this cell again.')

# Set all HuggingFace token env var names
# (different versions of the library use different names)
hf_token = os.getenv('HF_TOKEN', '')
if hf_token:
    os.environ['HF_TOKEN']               = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # older HF Hub versions
    os.environ['HUGGINGFACE_HUB_TOKEN']  = hf_token  # some transformers versions
    print(f'✅ HF_TOKEN set     : hf_****{hf_token[-4:]}')  # shows last 4 chars only
else:
    print('❌ HF_TOKEN is empty — check your .env file')

# Suppress other noisy warnings
os.environ['TOKENIZERS_PARALLELISM']       = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
print('✅ Warning filters  : applied')

✅ .env loaded from : c:\DSAI\ProductMatchAnalysis\product_matching_system_Basemodel\.env
✅ HF_TOKEN set     : hf_****ilTg
✅ Warning filters  : applied


In [3]:
# ── Standard imports (run AFTER the token cell above) ────────────────────────
import pandas as pd
import numpy as np
import time

# Import all logic from utils/matcher.py
from utils.matcher import (
    load_data,
    load_model,
    build_index,
    clean_text,
    find_match,
    compare_two_products,
    extract_title_from_url,
    extract_model_numbers,
    run_batch,
    generate_batch_excel,
    resolve_target_domain,
    MATCH_THRESHOLD,
    RETAILER_DOMAIN_MAP,
)

print('✅ All imports successful — ready to run experiments!')

✅ All imports successful — ready to run experiments!


---
## Section 1 — Data Loading & Inspection

In [4]:
# Change filename here to switch between Example 1 and Example 2
DATA_FILE = 'Matching List - Example 1 (1).xlsx'
DATA_PATH = os.path.join(project_root, 'data', DATA_FILE)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Data file not found: {DATA_PATH}\n'
        f'Check the filename matches what is in your data/ folder.'
    )

df = load_data(DATA_PATH)
print(f'✅ Loaded {len(df)} products from: {DATA_FILE}')
print(f'Columns: {list(df.columns)}')

✅ Loaded 76 products from: Matching List - Example 1 (1).xlsx
Columns: ['base_retailer_code', 'crawling title', 'url', 'target_retailer_code', 'url.1', 'Link', 'comment', 'Mindsite Comment', 'Unnamed: 8', 'url_slug', 'enriched_title', 'clean_title']


In [5]:
display(df)

,base_retailer_code,crawling title,url,target_retailer_code,url.1,Link,comment,Mindsite Comment,Unnamed: 8,url_slug,enriched_title,clean_title
0,HB,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,https://www.hepsiburada.com/sandisk-ultra-dual...,TY,www.trendyol.com,https://www.trendyol.com/sandisk/sddd3-128g-g4...,NaN,True,NaN,sddd3 128g g46 128gb black dual drive m3 0 mic...,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,sandisk ultra dual drive 128 gb otg m3 0 usb b...
1,HB,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,https://www.hepsiburada.com/redmi-13c-256-gb-8...,TY,www.trendyol.com,https://www.trendyol.com/xiaomi/redmi-13c-8-gb...,NaN,True,NaN,redmi 13c 8 gb ram 256 gb siyah cep telefonu x...,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,xiaomi redmi 13c 256 gb 8 gb ram xiaomi türkiy...
2,HB,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,https://www.hepsiburada.com/xiaomi-14t-256-gb-...,TY,www.trendyol.com,https://www.trendyol.com/xiaomi/14t-256-gb-mav...,NaN,mismatched,https://www.trendyol.com/xiaomi/14t-256-gb-mav...,14t 256 gb mavi xiaomi turkiye garantili,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,xiaomi 14t 256 gb 12 gb ram xiaomi türkiye gar...
3,HB,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.hepsiburada.com/samsung-1tb-mz-77e...,TY,www.trendyol.com,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,NaN,True,NaN,1tb mz 77e1t0bw 870 evo sata 3 0 560 530mb s 2...,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,samsung 1 tb mz 77e1t0bw 870 evo sata 3 0 560 ...
4,HB,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.hepsiburada.com/samsung-1tb-mz-77e...,TY,www.trendyol.com,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,NaN,True,NaN,1tb mz 77e1t0bw 870 evo sata 3 0 560 530mb s 2...,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,samsung 1 tb mz 77e1t0bw 870 evo sata 3 0 560 ...
...,...,...,...,...,...,...,...,...,...,...,...,...
71,MSM,Ülker Taç Kraker 3'lü Paket 228 G,https://www.migros.com.tr/ulker-tac-kraker-3lu...,A101,www.a101.com.tr,https://www.a101.com.tr/kapida/atistirmalik/ul...,NaN,mismatched,https://www.a101.com.tr/kapida/atistirmalik/ul...,ulker tac tuzlu kraker 3x76 g,Ülker Taç Kraker 3'lü Paket 228 G 3x76 tac tuz...,ülker taç kraker 3 lü paket 228 g 3x76 tac tuz...
72,AMZ_UAE,L’Oréal Paris L´Oreal Elvive Dream Long Straig...,https://www.amazon.ae/LOr%C3%A9al-Paris-L%C2%B...,ADW_KSA,www.al-dawaa.com,https://www.al-dawaa.com/ar/p/235139/loreal-pa...,NaN,True,NaN,loreal elvive oil replacement dream long strai...,L’Oréal Paris L´Oreal Elvive Dream Long Straig...,l oréal paris l oreal elvive dream long straig...
73,C4_UAE,Neutrogena Fresh And Clear Facial Wash Pink Gr...,https://www.carrefouruae.com/mafuae/en/face-cl...,ADW_KSA,www.al-dawaa.com,https://www.al-dawaa.com/ar/p/205925/neutrogen...,NaN,True,NaN,neutrogena fresh clear facial wash grapefruit ...,Neutrogena Fresh And Clear Facial Wash Pink Gr...,neutrogena fresh and clear facial wash pink gr...
74,NN_UAE,Natural Glow Face Wash 100ml,https://www.noon.com/uae-en/natural-glow-face-...,ADW_KSA,www.al-dawaa.com,https://www.al-dawaa.com/ar/p/211027/nivea-fac...,NaN,True,NaN,nivea face wash natural fairness vitamin e and...,Natural Glow Face Wash 100ml 100 and c e fair...,natural glow face wash 100ml 100 and c e fairn...


In [6]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   base_retailer_code    76 non-null     object
 1   crawling title        76 non-null     object
 2   url                   76 non-null     object
 3   target_retailer_code  76 non-null     object
 4   url.1                 76 non-null     object
 5   Link                  76 non-null     object
 6   comment               4 non-null      object
 7   Mindsite Comment      76 non-null     object
 8   Unnamed: 8            20 non-null     object
 9   url_slug              76 non-null     object
 10  enriched_title        76 non-null     object
 11  clean_title           76 non-null     object
dtypes: object(12)
memory usage: 7.3+ KB


None

In [7]:
# Inspect the enriched columns added by load_data()
df[['crawling title', 'Link', 'url_slug', 'enriched_title', 'clean_title']].head(10)

,crawling title,Link,url_slug,enriched_title,clean_title
0,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,https://www.trendyol.com/sandisk/sddd3-128g-g4...,sddd3 128g g46 128gb black dual drive m3 0 mic...,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,sandisk ultra dual drive 128 gb otg m3 0 usb b...
1,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,https://www.trendyol.com/xiaomi/redmi-13c-8-gb...,redmi 13c 8 gb ram 256 gb siyah cep telefonu x...,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,xiaomi redmi 13c 256 gb 8 gb ram xiaomi türkiy...
2,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,https://www.trendyol.com/xiaomi/14t-256-gb-mav...,14t 256 gb mavi xiaomi turkiye garantili,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,xiaomi 14t 256 gb 12 gb ram xiaomi türkiye gar...
3,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,1tb mz 77e1t0bw 870 evo sata 3 0 560 530mb s 2...,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,samsung 1 tb mz 77e1t0bw 870 evo sata 3 0 560 ...
4,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,1tb mz 77e1t0bw 870 evo sata 3 0 560 530mb s 2...,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,samsung 1 tb mz 77e1t0bw 870 evo sata 3 0 560 ...
5,"Smart Tank 580 Wireless All In One Printer, Pr...",https://www.noon.com/uae-en/smart-tank-580-wir...,,"Smart Tank 580 Wireless All In One Printer, Pr...",smart tank 580 wireless all in one printer pri...
6,"Smart Tank 580 Wireless All In One Printer, Pr...",https://www.noon.com/uae-en/smart-tank-580-wir...,,"Smart Tank 580 Wireless All In One Printer, Pr...",smart tank 580 wireless all in one printer pri...
7,"Smart Tank 580 Wireless All In One Printer, Pr...",https://www.noon.com/uae-en/smart-tank-580-wir...,,"Smart Tank 580 Wireless All In One Printer, Pr...",smart tank 580 wireless all in one printer pri...
8,Vichy Homme 72 Hour Deodorant Anti Perspirant ...,https://www.noon.com/saudi-en/homme-72h-anti-t...,,Vichy Homme 72 Hour Deodorant Anti Perspirant ...,vichy homme 72 hour deodorant anti perspirant ...
9,Epson C13S015633 Ink Ribbon Cartridge For Lq-3...,https://www.noon.com/uae-en/c13s015633-ink-rib...,,Epson C13S015633 Ink Ribbon Cartridge For Lq-3...,epson c13s015633 ink ribbon cartridge for lq 3...


In [8]:
# Check which target sites are in the dataset
from urllib.parse import urlparse

df['domain'] = df['Link'].apply(
    lambda x: urlparse(str(x)).netloc.replace('www.', '') if str(x).startswith('http') else 'unknown'
)
print('Products per domain:')
print(df['domain'].value_counts().to_string())

Products per domain:
domain
noon.com             9
mediamarkt.com.tr    9
amazon.ae            7
hepsiburada.com      6
trendyol.com         5
nahdionline.com      5
gratis.com           5
teknosa.com          4
takealot.com         4
migros.com.tr        4
carrefoursa.com      4
amazon.sa            4
al-dawaa.com         4
a101.com.tr          3
amazon.com.tr        2
carrefourksa.com     1


In [9]:
# Data quality check
print(f'Total rows:               {len(df)}')
print(f'Rows with valid links:    {df["Link"].str.startswith("http").sum()}')
print(f'Rows with empty titles:   {df["crawling title"].isna().sum()}')
print(f'Duplicate titles:         {df["crawling title"].duplicated().sum()}')
print(f'Avg title length (chars): {df["crawling title"].astype(str).str.len().mean():.1f}')

Total rows:               76
Rows with valid links:    76
Rows with empty titles:   0
Duplicate titles:         5
Avg title length (chars): 62.1


---
## Section 2 — Text Cleaning

In [10]:
test_cases = [
    'Samsung Galaxy S24 256GB - Blue',
    'سامسونج جالكسي S24 256 جيجا أزرق',
    'Xiaomi 14T 512GB Mavi (Türkçe)',
    'HP Laptop 15s-fq5037ne 256GB SSD',
    'Apple iPhone 15 Pro Max 1TB Natural Titanium',
]

print('Input → Cleaned output:')
print('-' * 60)
for t in test_cases:
    print(f'IN:  {t}')
    print(f'OUT: {clean_text(t)}')
    print()

Input → Cleaned output:
------------------------------------------------------------
IN:  Samsung Galaxy S24 256GB - Blue
OUT: samsung galaxy s24 256 gb blue

IN:  سامسونج جالكسي S24 256 جيجا أزرق
OUT: سامسونج جالكسي s24 256 جيجا أزرق

IN:  Xiaomi 14T 512GB Mavi (Türkçe)
OUT: xiaomi 14t 512 gb mavi türkçe

IN:  HP Laptop 15s-fq5037ne 256GB SSD
OUT: hp laptop 15s fq5037ne 256 gb ssd

IN:  Apple iPhone 15 Pro Max 1TB Natural Titanium
OUT: apple iphone 15 pro max 1 tb natural titanium



In [11]:
# Test model number extractor
test_titles = [
    'Samsung 256GB NFK37211',
    'HP 2S9L1AA laptop 15-inch',
    'Xiaomi Redmi Note 13C 128GB',
]
for t in test_titles:
    print(f'{t!r}  →  {extract_model_numbers(t)}')

'Samsung 256GB NFK37211'  →  {'256GB', 'NFK37211', 'SAMSUNG'}
'HP 2S9L1AA laptop 15-inch'  →  {'15INCH', 'LAPTOP', '2S9L1AA'}
'Xiaomi Redmi Note 13C 128GB'  →  {'NOTE', 'XIAOMI', 'REDMI', '128GB'}


---
## Section 3 — Model Loading & Embeddings

In [12]:
# First run downloads ~120MB, cached in .model_cache/ after that
# If HF_TOKEN was set correctly in Section 0, the unauthenticated warning will NOT appear

print('Loading multilingual SentenceTransformer...')
t0    = time.time()
model = load_model()
print(f'✅ Model loaded in {time.time()-t0:.1f}s')
print(f'   Embedding dimension: {model.get_sentence_embedding_dimension()}')

Loading multilingual SentenceTransformer...


BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded in 8.1s
   Embedding dimension: 384


In [13]:
# Cross-language similarity test
same_product = [
    'Samsung Galaxy S24 256GB Blue',
    'سامسونج جالكسي إس 24 256 جيجا أزرق',
    'Samsung Galaxy S24 256GB Mavi',
]
different = 'Apple iPhone 15 Pro 256GB Black'

embs = model.encode(same_product + [different], normalize_embeddings=True)

print('Cross-language similarity (higher = more similar):')
print(f'  English  vs Arabic:  {np.dot(embs[0], embs[1]):.4f}  ← should be HIGH (>0.80)')
print(f'  English  vs Turkish: {np.dot(embs[0], embs[2]):.4f}  ← should be HIGH (>0.80)')
print(f'  English  vs iPhone:  {np.dot(embs[0], embs[3]):.4f}  ← should be LOW  (<0.60)')

Cross-language similarity (higher = more similar):
  English  vs Arabic:  0.9156  ← should be HIGH (>0.80)
  English  vs Turkish: 0.8869  ← should be HIGH (>0.80)
  English  vs iPhone:  0.4648  ← should be LOW  (<0.60)


---
## Section 4 — FAISS Index

In [14]:
print(f'Building FAISS index from {len(df)} products...')
t0    = time.time()
index = build_index(df, model)
print(f'✅ Index built in {time.time()-t0:.1f}s')
print(f'   Vectors indexed : {index.ntotal}')
print(f'   Dimension       : {index.d}')

Building FAISS index from 76 products...


Batches: 100%|██████████| 3/3 [00:00<00:00,  5.34it/s]

✅ Index built in 0.6s
   Vectors indexed : 76
   Dimension       : 384


---
## Section 5 — Single Product Matching

In [15]:
# English query
query   = 'Samsung Galaxy S24 256GB'
results = find_match(query, model, index, df, threshold=0.70)

print(f'Query: "{query}"')
print(f'Domain filtered: {results.attrs.get("domain_filtered", False)}')
results[['crawling title', 'Similarity Score', 'Match Status', 'Link']].head(5)

Query: "Samsung Galaxy S24 256GB"
Domain filtered: False


,crawling title,Similarity Score,Match Status,Link
4,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,0.586488,❌ No Match,https://www.trendyol.com/samsung/1tb-mz-77e1t0...
3,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,0.586488,❌ No Match,https://www.trendyol.com/samsung/1tb-mz-77e1t0...
22,Samsung T7 Shield 1TB Mini USB 3.2 Bej Taşınab...,0.524476,❌ No Match,https://www.teknosa.com/samsung-t7-shield-mupe...
19,Xiaomi 13T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,0.481542,❌ No Match,https://www.teknosa.com/xiaomi-13t-12256gb-bla...
20,Xiaomi 14T 256 GB Siyah (Xiaomi Türkiye Garant...,0.435943,❌ No Match,https://www.teknosa.com/xiaomi-14t-12256-gb-si...


In [16]:
# Arabic query
query_ar    = 'سامسونج جالكسي 256 جيجا'
results_ar  = find_match(query_ar, model, index, df, threshold=0.65)
print(f'Query: "{query_ar}" (Arabic)')
results_ar[['crawling title', 'Similarity Score', 'Match Status']].head(5)

Query: "سامسونج جالكسي 256 جيجا" (Arabic)


,crawling title,Similarity Score,Match Status
4,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,0.463820,❌ No Match
3,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,0.463820,❌ No Match
19,Xiaomi 13T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,0.462401,❌ No Match
20,Xiaomi 14T 256 GB Siyah (Xiaomi Türkiye Garant...,0.444396,❌ No Match
22,Samsung T7 Shield 1TB Mini USB 3.2 Bej Taşınab...,0.428630,❌ No Match


In [17]:
# Partial query (<=4 words) — triggers keyword boost
query_partial   = 'Xiaomi 14T 256GB'
results_partial = find_match(query_partial, model, index, df, threshold=0.65)
print(f'Partial query: "{query_partial}"')
results_partial[['crawling title', 'Similarity Score', 'Match Status']].head(5)

Partial query: "Xiaomi 14T 256GB"


,crawling title,Similarity Score,Match Status
66,Xiaomi REDMI 14C 256 GB 8 GB Ram (Xiaomi Türki...,0.789568,✅ Match Found
20,Xiaomi 14T 256 GB Siyah (Xiaomi Türkiye Garant...,0.760672,✅ Match Found
2,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,0.755289,✅ Match Found
19,Xiaomi 13T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,0.748851,✅ Match Found
1,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,0.659834,✅ Match Found


In [18]:
# Threshold sensitivity
query = 'HP Laptop 15s 256GB'
print(f'Query: "{query}"')
print(f'{"Threshold":<12} {"Matches":<10} Top Score')
print('-' * 35)
for thresh in [0.60, 0.65, 0.70, 0.75, 0.80]:
    res       = find_match(query, model, index, df, threshold=thresh)
    matches   = (res['Similarity Score'] >= thresh).sum()
    top_score = res['Similarity Score'].iloc[0]
    print(f'{thresh:<12.2f} {matches}/5        {top_score:.4f}')

Query: "HP Laptop 15s 256GB"
Threshold    Matches    Top Score
-----------------------------------
0.60         0/5        0.4059
0.65         0/5        0.4059
0.70         0/5        0.4059
0.75         0/5        0.4059
0.80         0/5        0.4059


---
## Section 6 — Direct URL-to-URL Comparison

In [19]:
title_a = 'Samsung Galaxy S24 256GB Cobalt Violet'
title_b = 'سامسونج جالكسي S24 256 جيجا بنفسجي'  # Same product, Arabic
title_c = 'Apple iPhone 15 128GB Black'            # Different product

same = compare_two_products(title_a, title_b, model)
diff = compare_two_products(title_a, title_c, model)

print('Same product (EN vs AR):')
print(f'  Score: {same["score"]:.4f}  |  Match: {same["is_match"]}  |  Time: {same["elapsed"]}s')
print()
print('Different products:')
print(f'  Score: {diff["score"]:.4f}  |  Match: {diff["is_match"]}  |  Time: {diff["elapsed"]}s')

Same product (EN vs AR):
  Score: 0.7414  |  Match: False  |  Time: 0.0676s

Different products:
  Score: 0.3197  |  Match: False  |  Time: 0.0545s


---
## Section 7 — Domain Filtering

In [20]:
# Test resolve_target_domain with all input formats
test_inputs = ['TY', 'NN_KSA', 'www.noon.com', 'https://extra.com/product/123', 'trendyol.com', '']

print('resolve_target_domain() results:')
for inp in test_inputs:
    print(f'  {inp!r:45s} → {resolve_target_domain(inp)!r}')

resolve_target_domain() results:
  'TY'                                          → 'trendyol.com'
  'NN_KSA'                                      → 'noon.com'
  'www.noon.com'                                → 'noon.com'
  'https://extra.com/product/123'               → 'extra.com'
  'trendyol.com'                                → 'trendyol.com'
  ''                                            → ''


In [21]:
# Domain-filtered match
query      = 'Samsung Galaxy S24 256GB'
top_domain = df['domain'].value_counts().index[0]
print(f'Filtering to: {top_domain}')

results_f = find_match(query, model, index, df, threshold=0.65, target_filter=top_domain)
print(f'Filtered: {results_f.attrs.get("domain_filtered")}  |  Domain: {results_f.attrs.get("domain_used")}')
results_f[['crawling title', 'Similarity Score', 'Match Status', 'Link']].head()

Filtering to: noon.com
Filtered: True  |  Domain: noon.com


,crawling title,Similarity Score,Match Status,Link
8,Epson T6644 L100/L200 70ml Sarı Mürekkep,0.195790,❌ No Match,https://www.noon.com/uae-en/t6644-ecotank-ink-...
7,Epson T6644 L100/L200 70ml Sarı Mürekkep,0.195790,❌ No Match,https://www.noon.com/uae-en/t6644-ecotank-ink-...
6,"HP Smart Tank 670 All-in-One Yazıcı, Baskı, Ta...",0.180272,❌ No Match,https://www.noon.com/uae-en/smart-tank-670-all...
5,"HP Smart Tank 670 All-in-One Yazıcı, Baskı, Ta...",0.180272,❌ No Match,https://www.noon.com/uae-en/smart-tank-670-all...
2,"Smart Tank 580 Wireless All In One Printer, Pr...",0.168408,❌ No Match,https://www.noon.com/uae-en/smart-tank-580-wir...


---
## Section 8 — Batch Processing

In [22]:
test_batch = pd.DataFrame({
    'product_title': [
        'Samsung Galaxy S24 256GB',
        'Xiaomi 14T 512GB',
        'سامسونج جالكسي 256 جيجا',
        'HP Laptop 15s 256GB SSD',
        'Apple iPhone 15 128GB',
    ],
    'target_site': ['NN_KSA', 'TY', '', 'extra.com', ''],
})
test_batch

,product_title,target_site
0,Samsung Galaxy S24 256GB,NN_KSA
1,Xiaomi 14T 512GB,TY
2,سامسونج جالكسي 256 جيجا,
3,HP Laptop 15s 256GB SSD,extra.com
4,Apple iPhone 15 128GB,


In [23]:
def print_progress(current, total, message):
    print(f'  [{current}/{total}] {message}')

t0 = time.time()
batch_results = run_batch(
    test_batch, model, index, df,
    batch_threshold=0.68,
    global_target='',
    progress_callback=print_progress
)
elapsed = time.time() - t0
print(f'\n✅ Done in {elapsed:.1f}s  ({elapsed/len(test_batch):.2f}s per product)')

  [1/5] Processed: Samsung Galaxy S24 256GB...
  [2/5] Processed: Xiaomi 14T 512GB...
  [3/5] Processed: سامسونج جالكسي 256 جيجا...
  [4/5] Processed: HP Laptop 15s 256GB SSD...
  [5/5] Processed: Apple iPhone 15 128GB...

✅ Done in 0.6s  (0.12s per product)


In [24]:
batch_results[['Source Product', 'Target Website', 'Matched Product', 'Similarity Score', 'Match Status', 'AHT (seconds)']]

,Source Product,Target Website,Matched Product,Similarity Score,Match Status,AHT (seconds)
0,Samsung Galaxy S24 256GB,noon.com,Epson T6644 L100/L200 70ml Sarı Mürekkep,0.1958,No Match,0.166
1,Xiaomi 14T 512GB,trendyol.com,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,0.6651,No Match,0.320
2,سامسونج جالكسي 256 جيجا,N/A,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,0.4638,No Match,0.043
3,HP Laptop 15s 256GB SSD,extra.com,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,0.4155,No Match,0.043
4,Apple iPhone 15 128GB,N/A,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,0.4121,No Match,0.039


In [25]:
total   = len(batch_results)
matched = (batch_results['Match Status'] == 'Match Found').sum()
print(f'Total: {total} | Matched: {matched} | Rate: {matched/total*100:.1f}% | Avg AHT: {batch_results["AHT (seconds)"].mean():.3f}s')

Total: 5 | Matched: 0 | Rate: 0.0% | Avg AHT: 0.122s


In [26]:
output_path = os.path.join(project_root, 'data', 'test_batch_output.xlsx')
with open(output_path, 'wb') as f:
    f.write(generate_batch_excel(batch_results))
print(f'✅ Saved to {output_path}')

✅ Saved to c:\DSAI\ProductMatchAnalysis\product_matching_system_Basemodel\data\test_batch_output.xlsx


---
## Section 9 — Accuracy Evaluation (Ground Truth)

In [27]:
raw = pd.read_excel(DATA_PATH)
print(f'Raw columns: {list(raw.columns)}')
raw.head(3)

Raw columns: ['base_retailer_code', 'crawling title', 'url', 'target_retailer_code', 'url.1', 'Link', 'comment', 'Mindsite Comment', 'Unnamed: 8']


,base_retailer_code,crawling title,url,target_retailer_code,url.1,Link,comment,Mindsite Comment,Unnamed: 8
0,HB,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,https://www.hepsiburada.com/sandisk-ultra-dual...,TY,www.trendyol.com,https://www.trendyol.com/sandisk/sddd3-128g-g4...,NaN,True,NaN
1,HB,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,https://www.hepsiburada.com/redmi-13c-256-gb-8...,TY,www.trendyol.com,https://www.trendyol.com/xiaomi/redmi-13c-8-gb...,NaN,True,NaN
2,HB,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,https://www.hepsiburada.com/xiaomi-14t-256-gb-...,TY,www.trendyol.com,NaN,NaN,mismatched,https://www.trendyol.com/xiaomi/14t-256-gb-mav...


In [28]:
# ⚠️ Update these column names if they differ in your Excel file
SOURCE_TITLE_COL = 'crawling title'
CORRECT_LINK_COL = 'Link'

eval_df = raw[[SOURCE_TITLE_COL, CORRECT_LINK_COL]].dropna().copy()
eval_df = eval_df[eval_df[CORRECT_LINK_COL].astype(str).str.startswith('http')].head(20)
print(f'Evaluation set: {len(eval_df)} products with known correct links')

Evaluation set: 20 products with known correct links


In [29]:
correct, results_log = 0, []

for _, row in eval_df.iterrows():
    title        = str(row[SOURCE_TITLE_COL])
    ground_truth = str(row[CORRECT_LINK_COL]).strip()
    matches      = find_match(title, model, index, df, threshold=0.68)
    ai_link      = str(matches.iloc[0]['Link']).strip()
    is_correct   = (ai_link == ground_truth)
    correct     += int(is_correct)
    results_log.append({
        'Title':            title[:55],
        'Ground Truth URL': ground_truth[-55:],
        'AI Matched URL':   ai_link[-55:],
        'Score':            round(matches.iloc[0]['Similarity Score'], 4),
        'Correct':          '✅' if is_correct else '❌',
    })

print(f'📊 Accuracy: {correct}/{len(eval_df)} = {correct/len(eval_df)*100:.1f}%')
pd.DataFrame(results_log)

📊 Accuracy: 15/20 = 75.0%


,Title,Ground Truth URL,AI Matched URL,Score,Correct
0,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,ro-usb-flas-p-297488991?boutiqueId=61&merchant...,ro-usb-flas-p-297488991?boutiqueId=61&merchant...,0.9823,✅
1,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,siyah-cep-telefonu-xiaomi-turkiye-garantili-p-...,+xiaomi+t%C3%BCrkiye+garantili+siyah%2Caps%2C4...,0.9722,❌
2,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,sd-harddisk-p-121758595?boutiqueId=61&merchant...,sd-harddisk-p-121758595?boutiqueId=61&merchant...,0.9903,✅
3,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,sd-harddisk-p-121758595?boutiqueId=61&merchant...,sd-harddisk-p-121758595?boutiqueId=61&merchant...,0.9903,✅
4,"Smart Tank 580 Wireless All In One Printer, Pr...",f6a15adcbc&shareId=ed50955c-7427-4c2a-9bee-7cc...,f6a15adcbc&shareId=774e88a2-7e8d-42ed-9e8c-4d2...,1.0000,❌
5,"Smart Tank 580 Wireless All In One Printer, Pr...",f6a15adcbc&shareId=278dcba4-00e9-4534-811e-c79...,f6a15adcbc&shareId=774e88a2-7e8d-42ed-9e8c-4d2...,1.0000,❌
6,"Smart Tank 580 Wireless All In One Printer, Pr...",f6a15adcbc&shareId=774e88a2-7e8d-42ed-9e8c-4d2...,f6a15adcbc&shareId=774e88a2-7e8d-42ed-9e8c-4d2...,1.0000,✅
7,Epson C13S015633 Ink Ribbon Cartridge For Lq-3...,0bc8865c5e&shareId=4b8eb9f9-d6db-46ea-99b5-89d...,0bc8865c5e&shareId=4b8eb9f9-d6db-46ea-99b5-89d...,1.0000,✅
8,"HP Smart Tank 670 All-in-One Yazıcı, Baskı, Ta...",3c85e9cd8f&shareId=a6112616-318e-47f5-b560-3db...,3c85e9cd8f&shareId=90338d7b-e09a-473c-b242-0a9...,1.0000,❌
9,"HP Smart Tank 670 All-in-One Yazıcı, Baskı, Ta...",3c85e9cd8f&shareId=90338d7b-e09a-473c-b242-0a9...,3c85e9cd8f&shareId=90338d7b-e09a-473c-b242-0a9...,1.0000,✅


---
## Section 10 — AHT Benchmarking

In [30]:
benchmark_queries = [
    'Samsung Galaxy S24 256GB',
    'Xiaomi Redmi Note 13C 128GB',
    'HP Laptop 15s 512GB SSD',
    'Apple iPhone 15 Pro 1TB',
    'سامسونج جالكسي 256 جيجا',
    'Sony WH-1000XM5 Headphones',
    'LG OLED TV 55 inch 4K',
    'Dyson V15 Vacuum Cleaner',
    'Nespresso Vertuo Coffee Machine',
    'Nike Air Max 270 Size 42',
]

times = []
print(f'{"Time":<10} Query')
print('-' * 55)
for q in benchmark_queries:
    t0 = time.time()
    find_match(q, model, index, df, threshold=0.70)
    elapsed = time.time() - t0
    times.append(elapsed)
    print(f'{elapsed:.3f}s     {q}')

print(f'\n  Avg: {np.mean(times):.3f}s  |  Min: {np.min(times):.3f}s  |  Max: {np.max(times):.3f}s')

Time       Query
-------------------------------------------------------
0.040s     Samsung Galaxy S24 256GB
0.038s     Xiaomi Redmi Note 13C 128GB
0.034s     HP Laptop 15s 512GB SSD
0.034s     Apple iPhone 15 Pro 1TB
0.034s     سامسونج جالكسي 256 جيجا
0.035s     Sony WH-1000XM5 Headphones
0.090s     LG OLED TV 55 inch 4K
0.035s     Dyson V15 Vacuum Cleaner
0.040s     Nespresso Vertuo Coffee Machine
0.030s     Nike Air Max 270 Size 42

  Avg: 0.041s  |  Min: 0.030s  |  Max: 0.090s


In [31]:
# ← Adjust these two numbers to match your team's real situation
manual_aht_minutes = 5
daily_products     = 100

manual_sec         = manual_aht_minutes * 60
ai_sec             = np.mean(times)
reduction_pct      = (manual_sec - ai_sec) / manual_sec * 100
daily_savings_mins = (manual_sec - ai_sec) * daily_products / 60

print('📊 AHT Reduction Summary  (use this in your client presentation)')
print('=' * 52)
print(f'  Manual AHT      : {manual_sec}s ({manual_aht_minutes} min/product)')
print(f'  AI AHT          : {ai_sec:.3f}s/product')
print(f'  AHT Reduction   : {reduction_pct:.1f}%')
print(f'  Daily savings   : {daily_savings_mins:.0f} min ({daily_savings_mins/60:.1f} hrs)')
print(f'  (based on {daily_products} products/day)')

📊 AHT Reduction Summary  (use this in your client presentation)
  Manual AHT      : 300s (5 min/product)
  AI AHT          : 0.041s/product
  AHT Reduction   : 100.0%
  Daily savings   : 500 min (8.3 hrs)
  (based on 100 products/day)


---
## Section 11 — 🧪 Quick Test: Single Product or Batch File

Use these cells to **quickly test the full pipeline** without running the Streamlit app.

| Cell | What it does |
|---|---|
| 11-A | Single product title search — paste any title |
| 11-B | Single product via URL — auto-extracts title from URL |
| 11-C | Batch test from CSV or Excel file |
| 11-D | Save batch results to Excel |

> ⚠️ **Make sure Sections 0–4 are already run** before running any cell here  
> (i.e. `df`, `model`, `index` must be loaded)

### Cell 11-A — Single Product Title Search
Paste any product title below (Arabic, English, Turkish — all supported).

In [32]:
import time

# ── ✏️  EDIT THESE TWO LINES ───────────────────────────────────────────────────
QUERY         = "Samsung Galaxy S24 256GB"   # ← paste your product title here
TARGET_FILTER = ""                           # ← domain/code like 'noon.com' or 'NN_KSA', or leave empty
THRESHOLD     = 0.70                         # ← lower = more results, higher = stricter
# ──────────────────────────────────────────────────────────────────────────────

t0      = time.time()
results = find_match(
    QUERY, model, index, df,
    threshold=THRESHOLD,
    target_filter=TARGET_FILTER if TARGET_FILTER else None,
)
elapsed = time.time() - t0

print(f"🔍 Query       : {QUERY}")
print(f"🌐 Target      : {TARGET_FILTER or '(all sites)'}")
print(f"⏱  AHT         : {elapsed:.3f}s")
print(f"📊 Threshold   : {THRESHOLD}")
print(f"🔎 Domain used : {results.attrs.get('domain_used', 'none')}")
print()

# Show top 5 results as a clean table
display_cols = ["crawling title", "Link", "Similarity Score", "Match Status"]
display(results[display_cols].reset_index(drop=True))

🔍 Query       : Samsung Galaxy S24 256GB
🌐 Target      : (all sites)
⏱  AHT         : 0.036s
📊 Threshold   : 0.7
🔎 Domain used : 



,crawling title,Link,Similarity Score,Match Status
0,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,0.586488,❌ No Match
1,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,0.586488,❌ No Match
2,Samsung T7 Shield 1TB Mini USB 3.2 Bej Taşınab...,https://www.teknosa.com/samsung-t7-shield-mupe...,0.524476,❌ No Match
3,Xiaomi 13T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,https://www.teknosa.com/xiaomi-13t-12256gb-bla...,0.481542,❌ No Match
4,Xiaomi 14T 256 GB Siyah (Xiaomi Türkiye Garant...,https://www.teknosa.com/xiaomi-14t-12256-gb-si...,0.435943,❌ No Match


### Cell 11-B — Single Product via URL
Paste a source product URL — the title will be auto-extracted, then matched.

In [33]:
# ── ✏️  EDIT THESE LINES ───────────────────────────────────────────────────────
SOURCE_URL    = "https://www.hepsiburada.com/sandisk-ultra-dual-drive-m3-0-128gb-otg-usb-bellek-hbv00000u3vv6-pm-HBC00001S7WN"  # ← paste product URL
TARGET_FILTER = "trendyol.com"  # ← leave empty to search all sites
THRESHOLD     = 0.70
# ──────────────────────────────────────────────────────────────────────────────

print(f"📡 Fetching title from URL...")
print(f"   {SOURCE_URL[:80]}...")
print()

extraction = extract_title_from_url(SOURCE_URL)

if extraction["success"]:
    title = extraction["title"]
    print(f"✅ Extracted title  : {title}")
    print(f"   Source method    : {extraction['source']}")
    print()

    t0      = time.time()
    results = find_match(
        title, model, index, df,
        threshold=THRESHOLD,
        target_filter=TARGET_FILTER if TARGET_FILTER else None,
    )
    elapsed = time.time() - t0

    print(f"⏱  AHT             : {elapsed:.3f}s")
    print(f"🌐 Domain filter   : {results.attrs.get('domain_used', 'none (all sites)')}")
    print()
    display_cols = ["crawling title", "Link", "Similarity Score", "Match Status"]
    display(results[display_cols].reset_index(drop=True))

else:
    print(f"❌ Could not extract title: {extraction['error']}")
    print()
    print("ℹ️  This is expected for JavaScript-rendered sites (noon, jarir).")
    print("   Try copying the product title manually and using Cell 11-A instead.")

📡 Fetching title from URL...
   https://www.hepsiburada.com/sandisk-ultra-dual-drive-m3-0-128gb-otg-usb-bellek-h...

✅ Extracted title  : Hepsiburada
   Source method    : page_title

⏱  AHT             : 0.109s
🌐 Domain filter   : trendyol.com



,crawling title,Link,Similarity Score,Match Status
0,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,https://www.trendyol.com/xiaomi/14t-256-gb-mav...,0.100737,❌ No Match
1,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,https://www.trendyol.com/xiaomi/redmi-13c-8-gb...,0.098490,❌ No Match
2,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,0.026214,❌ No Match
3,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,https://www.trendyol.com/samsung/1tb-mz-77e1t0...,0.026214,❌ No Match
4,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,https://www.trendyol.com/sandisk/sddd3-128g-g4...,-0.023525,❌ No Match


### Cell 11-C — Batch Test from CSV or Excel File
Provide a path to a file with product titles (and optional URLs/targets). Results are displayed in the notebook.

In [34]:
import os

# ── ✏️  EDIT THESE LINES ───────────────────────────────────────────────────────
BATCH_FILE     = os.path.join(project_root, 'data', 'Matching List - Example 1 (1).xlsx')
                 # ↑ change to your file path, e.g. '../data/my_products.csv'
GLOBAL_TARGET  = ""     # ← fallback target for rows that have no target column
BATCH_THRESHOLD = 0.70  # ← match threshold for batch
# ──────────────────────────────────────────────────────────────────────────────

if not os.path.exists(BATCH_FILE):
    print(f"❌ File not found: {BATCH_FILE}")
    print("   Update BATCH_FILE path above.")
else:
    # ── Load file ──────────────────────────────────────────────────────────────
    print(f"📂 Loading: {os.path.basename(BATCH_FILE)} ...")
    if BATCH_FILE.lower().endswith('.csv'):
        raw_batch = pd.read_csv(BATCH_FILE)
    else:
        raw_batch = pd.read_excel(BATCH_FILE)
    print(f"✅ Loaded {len(raw_batch)} rows")
    print(f"   Columns: {list(raw_batch.columns)}")
    print()

    # ── Auto-detect columns ─────────────────────────────────────────────────────
    cols_lower = {c.lower().replace(' ', '_'): c for c in raw_batch.columns}
    title_col  = next((cols_lower[k] for k in
                       ['crawling_title', 'title', 'product_title', 'product', 'name']
                       if k in cols_lower), None)
    url_col    = next((cols_lower[k] for k in
                       ['url', 'source_url', 'link', 'source']
                       if k in cols_lower), None)
    target_col = next((cols_lower[k] for k in
                       ['target_retailer_code', 'target_site', 'target', 'url.1', 'url_1']
                       if k in cols_lower), None)

    print(f"Auto-detected columns:")
    print(f"  Title  : {title_col  or '❌ not found — rename to crawling title or title'}")
    print(f"  URL    : {url_col    or '(not found — optional)'}")
    print(f"  Target : {target_col or '(not found — optional)'}")
    print()

    if title_col is None and url_col is None:
        print("❌ Cannot proceed — no title or URL column found.")
        print("   Please rename your title column to 'title' or 'crawling title'.")
    else:
        # ── Build input_df ──────────────────────────────────────────────────────
        input_df = pd.DataFrame()
        if title_col:  input_df['product_title'] = raw_batch[title_col].astype(str).str.strip()
        if url_col:    input_df['source_url']    = raw_batch[url_col].astype(str).str.strip()
        if target_col: input_df['target_site']   = raw_batch[target_col].astype(str).str.strip()

        # ── Progress callback for notebook ─────────────────────────────────────
        def nb_progress(current, total, msg):
            if current % max(1, total // 10) == 0 or current == total:
                pct = current / total * 100
                print(f"  [{pct:5.1f}%] {current}/{total}  —  {msg[:60]}")

        print(f"⚙️  Running batch matching on {len(input_df)} products...")
        print()
        t0             = time.time()
        batch_results  = run_batch(
            input_df, model, index, df,
            batch_threshold=BATCH_THRESHOLD,
            global_target=GLOBAL_TARGET,
            progress_callback=nb_progress,
        )
        total_time = time.time() - t0

        # ── Summary ─────────────────────────────────────────────────────────────
        total_r   = len(batch_results)
        matched_r = len(batch_results[batch_results['Match Status'] == 'Match Found'])
        avg_aht   = batch_results['AHT (seconds)'].mean()

        print()
        print("=" * 55)
        print(f"  ✅ Batch complete!")
        print(f"  Total products   : {total_r}")
        print(f"  Matched          : {matched_r}  ({matched_r/total_r*100:.1f}%)")
        print(f"  No Match         : {total_r - matched_r}")
        print(f"  Total time       : {total_time:.2f}s")
        print(f"  Avg AHT/product  : {avg_aht:.4f}s")
        print("=" * 55)
        print()

        # ── Display results table ────────────────────────────────────────────────
        display_cols = [
            'Row', 'Source Product', 'Target Website',
            'Matched Product', 'Similarity Score', 'Match Status', 'AHT (seconds)'
        ]
        display(batch_results[display_cols])
        
        # Store in notebook scope so Cell 11-D can save it
        _last_batch_results = batch_results

📂 Loading: Matching List - Example 1 (1).xlsx ...
✅ Loaded 77 rows
   Columns: ['base_retailer_code', 'crawling title', 'url', 'target_retailer_code', 'url.1', 'Link', 'comment', 'Mindsite Comment', 'Unnamed: 8']

Auto-detected columns:
  Title  : crawling title
  URL    : url
  Target : target_retailer_code

⚙️  Running batch matching on 77 products...

  [  9.1%] 7/77  —  Processed: Smart Tank 580 Wireless All In One Printer, Print
  [ 18.2%] 14/77  —  Processed: Epson T6644 L100/L200 70ml Sarı Mürekkep...
  [ 27.3%] 21/77  —  Processed: Xiaomi 14T 256 GB Siyah (Xiaomi Türkiye Garantili
  [ 36.4%] 28/77  —  Processed: Colgate Optic White Toothpaste 75ml...
  [ 45.5%] 35/77  —  Processed: Ülker Kekstra Jolebol Çileki Mini 150 Gr...
  [ 54.5%] 42/77  —  Processed: logitech Wave Keys Kablosuz Bluetooth Dolgulu Avu
  [ 63.6%] 49/77  —  Processed: Maybelline New York Lash Sensational Yelpaze Etki
  [ 72.7%] 56/77  —  Processed: Colgate Advanced Whitening Toothpaste 125ml...
  [ 81.8%] 63/

,Row,Source Product,Target Website,Matched Product,Similarity Score,Match Status,AHT (seconds)
0,1,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,trendyol.com,Sandisk Ultra Dual Drive 128GB OTG M3.0 Usb Be...,0.9823,Match Found,0.145
1,2,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,trendyol.com,Xiaomi Redmi 13C 256 GB 8 GB Ram (Xiaomi Türki...,0.8523,Match Found,0.197
2,3,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,trendyol.com,Xiaomi 14T 256 GB 12 GB Ram (Xiaomi Türkiye Ga...,0.9217,Match Found,0.183
3,4,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,trendyol.com,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,0.9903,Match Found,0.188
4,5,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,trendyol.com,Samsung 1TB MZ-77E1T0BW 870 Evo Sata 3.0 560-5...,0.9903,Match Found,0.188
...,...,...,...,...,...,...,...
72,73,Ülker Taç Kraker 3'lü Paket 228 G,a101.com.tr,Ülker Taç Kraker 3'lü Paket 228 G,0.8631,Match Found,0.103
73,74,L’Oréal Paris L´Oreal Elvive Dream Long Straig...,al-dawaa.com,L’Oréal Paris L´Oreal Elvive Dream Long Straig...,0.9667,Match Found,0.099
74,75,Neutrogena Fresh And Clear Facial Wash Pink Gr...,al-dawaa.com,Neutrogena Fresh And Clear Facial Wash Pink Gr...,0.9944,Match Found,0.093
75,76,Natural Glow Face Wash 100ml,al-dawaa.com,Natural Glow Face Wash 100ml,0.8839,Match Found,0.096


### Cell 11-D — Save Batch Results to Excel
Run this after Cell 11-C to save the results.

In [35]:
# ── ✏️  EDIT OUTPUT PATH if needed ─────────────────────────────────────────────
OUTPUT_FILE = os.path.join(project_root, 'batch_test_results.xlsx')
# ──────────────────────────────────────────────────────────────────────────────

try:
    excel_bytes = generate_batch_excel(_last_batch_results)
    with open(OUTPUT_FILE, 'wb') as f:
        f.write(excel_bytes)
    print(f"✅ Results saved to: {OUTPUT_FILE}")
    print(f"   {len(_last_batch_results)} rows written (with clickable links + Summary tab)")
except NameError:
    print("❌ No batch results found. Run Cell 11-C first.")

✅ Results saved to: c:\DSAI\ProductMatchAnalysis\product_matching_system_Basemodel\batch_test_results.xlsx
   77 rows written (with clickable links + Summary tab)
